# Kolmogorov-Arnold Networks in Python from Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/kolmogorov_arnold_networks.ipynb)

An MLP puts a fixed activation on every node and one learned number on every edge. A Kolmogorov-Arnold network does the opposite: each edge carries a learned univariate function and each node only adds.

This notebook builds one in NumPy with hand-derived gradients, fits `f(x, y) = exp(sin(pi x) + y^2)` with 39 parameters, and then reads the learned edge functions back out to find `sin(pi x)`, `y^2` and `exp(.)` sitting in them.

Everything here runs on a CPU in a couple of minutes. No GPU, no framework.

Companion post: [Kolmogorov-Arnold Networks in Python from Scratch](https://sesen.ai/blog/kolmogorov-arnold-networks-python-from-scratch)

## 1. A cubic B-spline basis

Each edge function is a weighted sum of B-spline basis functions. A B-spline basis function is a bump with **local support**: non-zero over a few grid intervals, exactly zero everywhere else. That locality is what makes the architecture work, because moving one control point reshapes the curve near one knot and leaves the rest alone.

The Cox-de Boor recursion builds degree-p basis functions from degree-(p-1) ones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [ ]:
class BSpline:
    """Cubic B-spline basis over a uniform grid, by Cox-de Boor recursion."""

    def __init__(self, grid, degree=3):
        h = grid[1] - grid[0]
        self.grid, self.degree = grid, degree
        self.knots = np.concatenate([grid[0] - h * np.arange(degree, 0, -1), grid,
                                     grid[-1] + h * np.arange(1, degree + 1)])
        self.n_basis = len(self.knots) - degree - 1

    def _b(self, x, p):
        t = self.knots
        b = ((x[..., None] >= t[:-1]) & (x[..., None] < t[1:])).astype(float)
        for d in range(1, p + 1):
            la, ra = t[d:-1] - t[:-d - 1], t[d + 1:] - t[1:-d]
            b = (np.where(la > 0, (x[..., None] - t[:-d - 1]) / np.where(la > 0, la, 1), 0) * b[..., :-1]
                 + np.where(ra > 0, (t[d + 1:] - x[..., None]) / np.where(ra > 0, ra, 1), 0) * b[..., 1:])
        return b

    def __call__(self, x):
        return self._b(np.clip(x, self.knots[0], self.knots[-1] - 1e-9), self.degree)

    def deriv(self, x):
        p, t = self.degree, self.knots
        lo = self._b(np.clip(x, t[0], t[-1] - 1e-9), p - 1)
        da, db = t[p:-1] - t[:-p - 1], t[p + 1:] - t[1:-p]
        return (np.where(da > 0, p / np.where(da > 0, da, 1), 0) * lo[..., :-1]
                - np.where(db > 0, p / np.where(db > 0, db, 1), 0) * lo[..., 1:])

In [ ]:
# Look at the basis before using it. Each bump is non-zero over a few intervals
# and exactly zero elsewhere, which is the locality the whole architecture rests on.
basis = BSpline(np.linspace(-1, 1, 9))
xs = np.linspace(-1, 1, 400)
B = basis(xs)

fig, ax = plt.subplots(figsize=(8, 3.2))
for k in range(basis.n_basis):
    ax.plot(xs, B[:, k], linewidth=1.4)
ax.set_title(f"{basis.n_basis} cubic B-spline basis functions on a grid of 8 intervals")
ax.set_xlabel("x")
ax.grid(alpha=0.25)
plt.show()

print("basis functions:", basis.n_basis, "= grid intervals (8) + degree (3)")
print("they sum to 1 in the interior:", np.round(B[200].sum(), 6))

## 2. A KAN layer

The layer computes `y_j = sum_i phi_ji(x_i)`, where each edge function is

```
phi(x) = w_b * silu(x) + w_s * sum_k c_k B_k(x)
```

The spline control points `c_k` are ordinary weights. The silu residual gives every edge a sensible non-zero starting shape, which matters because a spline initialised near zero has little gradient signal to work with.

Gradients are derived by hand. Since phi is linear in `c`, the gradient with respect to a control point is just that basis function's value; passing gradient back to the input needs the spline derivative, and B-splines differentiate into lower-degree B-splines.

In [ ]:
silu = lambda x: x / (1 + np.exp(-np.clip(x, -60, 60)))


class KANLayer:
    """y_j = sum_i phi_ji(x_i), with phi = w_b*silu(x) + w_s*sum_k c_k B_k(x)."""

    def __init__(self, n_in, n_out, grid, rng):
        self.basis = BSpline(grid)
        self.c = rng.normal(0, 0.1, (n_out, n_in, self.basis.n_basis)) / np.sqrt(n_in)
        self.ws = np.ones((n_out, n_in)) / np.sqrt(n_in)
        self.wb = rng.normal(0, 1, (n_out, n_in)) / np.sqrt(n_in)

    @property
    def n_params(self):
        return self.c.size + self.ws.size + self.wb.size

    def forward(self, x):
        self.x, self.B = x, self.basis(x)
        self.spl = np.einsum("nib,oib->noi", self.B, self.c)
        return (self.ws * self.spl).sum(-1) + silu(x) @ self.wb.T

    def backward(self, g):                       # g = dL/dy, shape (N, n_out)
        self.gc = np.einsum("no,oi,nib->oib", g, self.ws, self.B)
        self.gws = np.einsum("no,noi->oi", g, self.spl)
        self.gwb = g.T @ silu(self.x)
        s = 1 / (1 + np.exp(-np.clip(self.x, -60, 60)))
        d = np.einsum("nib,oib->noi", self.basis.deriv(self.x), self.c)
        return np.einsum("no,oi,noi->ni", g, self.ws, d) + (g @ self.wb) * s * (1 + self.x * (1 - s))

    def phi(self, i, j, xs):
        """The learned function sitting on the edge from input i to output j."""
        return self.wb[j, i] * silu(xs) + self.ws[j, i] * (self.basis(xs) * self.c[j, i]).sum(-1)

In [ ]:
class KAN:
    def __init__(self, widths, grid_points=8, span=(-1.2, 1.2), seed=0):
        rng = np.random.default_rng(seed)
        self.layers = [KANLayer(a, b, np.linspace(*span, grid_points + 1), rng)
                       for a, b in zip(widths[:-1], widths[1:])]

    @property
    def n_params(self):
        return sum(l.n_params for l in self.layers)

    def forward(self, x):
        for l in self.layers:
            x = l.forward(x)
        return x

    def update_grids(self, x):
        """Re-centre each layer's grid on the values it sees.

        Layer one's inputs are the data and hold still. Deeper layers see the
        previous layer's output, which moves during training; leave the grid fixed
        and the signal drifts off the spline's support, where no gradient flows.
        """
        h = x
        for l in self.layers:
            lo, hi = h.min(), h.max()
            pad = 0.1 * max(hi - lo, 1e-3)
            l.basis = BSpline(np.linspace(lo - pad, hi + pad, len(l.basis.grid)))
            h = l.forward(h)

    def refine_grid(self, x, new_points):
        """Grid extension: least-squares fit a finer basis to the curve already learned."""
        for l in self.layers:
            old = l.basis
            xs = np.linspace(old.grid[0], old.grid[-1], 400)
            target = np.einsum("xb,oib->oix", old(xs), l.c)
            new = BSpline(np.linspace(old.grid[0], old.grid[-1], new_points + 1), old.degree)
            coef, *_ = np.linalg.lstsq(new(xs), target.reshape(-1, len(xs)).T, rcond=None)
            l.basis, l.c = new, coef.T.reshape(l.c.shape[0], l.c.shape[1], new.n_basis)

    def fit(self, x, y, steps=2500, lr=0.02, grid_every=200):
        m_s = [[np.zeros_like(p) for p in (l.c, l.ws, l.wb)] for l in self.layers]
        v_s = [[np.zeros_like(p) for p in (l.c, l.ws, l.wb)] for l in self.layers]
        self.history = []
        for t in range(1, steps + 1):
            if grid_every and t % grid_every == 1 and t > 1:
                self.update_grids(x)
            out = self.forward(x)
            self.history.append(float(np.mean((out - y) ** 2)))
            g = 2 * (out - y) / len(x)
            for l in reversed(self.layers):
                g = l.backward(g)
            for li, l in enumerate(self.layers):
                for k, (p, gr) in enumerate(zip((l.c, l.ws, l.wb), (l.gc, l.gws, l.gwb))):
                    m, v = m_s[li][k], v_s[li][k]
                    m *= 0.9; m += 0.1 * gr
                    v *= 0.999; v += 0.001 * gr ** 2
                    p -= lr * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)
        return self

## 3. Fit a formula with 39 parameters

The target is the example from the KAN paper: `f(x, y) = exp(sin(pi x) + y^2)`.

A `[2, 1, 1]` KAN has three edges. With 8 grid intervals and cubic splines that is 11 coefficients plus 2 scalars per edge, so 39 parameters in total.

Note the grid re-centring inside `fit`. Layer one's inputs are the data and hold still, but layer two's inputs are layer one's outputs, which move throughout training. Leave the grid fixed and the signal drifts off the end of the spline's support, where every basis function is zero and no gradient flows.

In [ ]:
rng = np.random.default_rng(0)
X = rng.uniform(-1, 1, (1000, 2))
y_raw = np.exp(np.sin(np.pi * X[:, 0]) + X[:, 1] ** 2)[:, None]
y_mu, y_sd = y_raw.mean(), y_raw.std()
y = (y_raw - y_mu) / y_sd

kan = KAN([2, 1, 1], grid_points=8, seed=1).fit(X, y, steps=2500)

print("parameters :", kan.n_params)
print("train RMSE :", round(float(np.sqrt(np.mean((kan.forward(X) - y) ** 2))), 5))

## 4. Read the edges

This is the part an MLP cannot do. The network was given `(x, y)` pairs and a target. Nobody told it the target factorises, nor which variable belongs in which term.

The decomposition is unique only up to a shared affine transformation: multiply both first-layer edges by 3, add 7, and let the second layer undo it, and the output is identical. So the honest test scores shape, not absolute values, by fitting an affine map between the learned edge and the candidate term and reporting R-squared.

In [ ]:
def affine_r2(learned, truth):
    """R^2 of the best affine map from `truth` onto `learned`.

    The decomposition is unique only up to a shared affine transformation: scale
    both first-layer edges and let the second layer undo it, and the output is
    unchanged. So compare shape, not absolute values.
    """
    A = np.column_stack([truth, np.ones_like(truth)])
    resid = learned - A @ np.linalg.lstsq(A, learned, rcond=None)[0]
    return 1 - (resid ** 2).sum() / ((learned - learned.mean()) ** 2).sum()


xs = np.linspace(-1, 1, 300)
phi_x = kan.layers[0].phi(0, 0, xs)
phi_y = kan.layers[0].phi(1, 0, xs)

print("edge from x  vs  sin(pi x) :", round(affine_r2(phi_x, np.sin(np.pi * xs)), 6))
print("edge from y  vs  y^2       :", round(affine_r2(phi_y, xs ** 2), 6))
print()
print("and against the wrong term, to check the match is specific:")
print("edge from x  vs  y^2       :", round(affine_r2(phi_x, xs ** 2), 6))
print("edge from y  vs  sin(pi x) :", round(affine_r2(phi_y, np.sin(np.pi * xs)), 6))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, learned, truth, name, var in [
        (axes[0], phi_x, np.sin(np.pi * xs), "sin(pi x)", "x"),
        (axes[1], phi_y, xs ** 2, "y^2", "y")]:
    A = np.column_stack([truth, np.ones_like(truth)])
    fitted = A @ np.linalg.lstsq(A, learned, rcond=None)[0]
    ax.plot(xs, fitted, linewidth=3.2, alpha=0.5, label=f"true term {name}")
    ax.plot(xs, learned, linewidth=1.8, label="learned edge function")
    ax.set_title(f"R^2 = {affine_r2(learned, truth):.6f}")
    ax.set_xlabel(var)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
plt.show()

## 5. The scale is free, the ratio is not

Run five seeds. Every one recovers the terms, and the amplitudes differ every time, which is the affine freedom showing up. What does not vary is the **ratio** between the two edges' amplitudes. `sin(pi x)` has range 2 over [-1, 1] and `y^2` has range 1, so the true ratio is exactly 2, and the network gets it to four significant figures on every seed while choosing the overall scale freely.

In [ ]:
rows = []
for seed in range(5):
    k = KAN([2, 1, 1], grid_points=8, seed=seed).fit(X, y, steps=2500)
    px = k.layers[0].phi(0, 0, xs)
    py = k.layers[0].phi(1, 0, xs)
    rows.append((seed, affine_r2(px, np.sin(np.pi * xs)), affine_r2(py, xs ** 2),
                 np.ptp(px), np.ptp(py), np.ptp(px) / np.ptp(py)))

print(f"{'seed':>5}{'R2(sin)':>11}{'R2(sq)':>11}{'amp(x)':>9}{'amp(y)':>9}{'ratio':>9}")
for s, a, b, ax_, ay, r in rows:
    print(f"{s:>5}{a:>11.6f}{b:>11.6f}{ax_:>9.3f}{ay:>9.3f}{r:>9.4f}")
print()
print("The amplitudes are free. The ratio is not: sin(pi x) has range 2 over")
print("[-1, 1] and y^2 has range 1, so the true ratio is exactly 2.")

## 6. Grid extension

Splines support an operation with no MLP analogue. Once an edge function is learned on a coarse grid, you can least-squares fit a finer spline to the curve the coarse one already describes, and carry on training. The curve survives the change of representation; only the resolution available to it improves.

The control is the interesting part: training the fine grid from scratch for the same total number of steps lands several times worse.

In [ ]:
# A fresh generator, so this section reproduces the numbers in the post rather
# than depending on how much randomness the cells above happened to consume.
gen = np.random.default_rng(0)
f = lambda x: np.sin(2 * np.pi * x) + 0.3 * np.sin(10 * np.pi * x)
xtr = np.sort(gen.uniform(-1, 1, 600))[:, None]
ytr = f(xtr)
xte = np.linspace(-1, 1, 500)[:, None]
yte = f(xte)
rmse = lambda m: float(np.sqrt(np.mean((m.forward(xte) - yte) ** 2)))

coarse = KAN([1, 3, 1], grid_points=5, seed=0).fit(xtr, ytr, steps=1500)
before = rmse(coarse)
print(f"grid 5, {coarse.n_params:3d} params                     test RMSE {before:.5f}")

coarse.refine_grid(xtr, new_points=20)
after = rmse(coarse)
# The refit is a change of basis, not a retrain, so report how much it actually
# moved rather than asserting it moved nothing.
print(f"refit onto grid 20 (no training)          test RMSE {after:.5f}"
      f"   <- {abs(after - before) / before:.1%} change")

coarse.fit(xtr, ytr, steps=1500, lr=0.01, grid_every=0)
print(f"after 1500 more steps, {coarse.n_params:3d} params      test RMSE {rmse(coarse):.5f}")

scratch = KAN([1, 3, 1], grid_points=20, seed=0).fit(xtr, ytr, steps=3000)
print(f"grid 20 from scratch, same total budget   test RMSE {rmse(scratch):.5f}")
print()
print("Refining beats starting fine, at equal total cost.")

## 7. Where KANs lose

A post that only showed the wins would be an advert. Two costs are structural.

**Speed.** An MLP edge is one multiply. A KAN edge evaluates `G+k` basis functions through a multi-pass recursion, then a weighted sum, and the backward pass runs the recursion again for the derivative.

**Data without formulas.** On 8x8 handwritten digits there is no symbolic structure for the edges to find, and the KAN is worse on accuracy, parameters and time at once. Pixels are not `sin(pi x)`.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import time


class MLP:
    """Matched baseline: fixed tanh on the node, learned scalar on the edge."""

    def __init__(self, widths, seed=0):
        r = np.random.default_rng(seed)
        self.W = [r.normal(0, np.sqrt(2 / a), (a, b)) for a, b in zip(widths[:-1], widths[1:])]
        self.b = [np.zeros(b) for b in widths[1:]]

    @property
    def n_params(self):
        return sum(w.size for w in self.W) + sum(b.size for b in self.b)

    def forward(self, x):
        self.a = [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.a[-1] @ W + b
            self.a.append(z if i == len(self.W) - 1 else np.tanh(z))
        return self.a[-1]

    def backward(self, g):
        self.gW, self.gb = [None] * len(self.W), [None] * len(self.b)
        for i in reversed(range(len(self.W))):
            self.gW[i], self.gb[i] = self.a[i].T @ g, g.sum(0)
            if i:
                g = (g @ self.W[i].T) * (1 - self.a[i] ** 2)


def fit_classifier(model, X, yi, steps=600, lr=0.01, batch=128, seed=0):
    r = np.random.default_rng(seed)
    if isinstance(model, KAN):
        params = lambda: [(p, g) for l in model.layers
                          for p, g in ((l.c, l.gc), (l.ws, l.gws), (l.wb, l.gwb))]
        shapes = [p for l in model.layers for p in (l.c, l.ws, l.wb)]
    else:
        params = lambda: list(zip(model.W, model.gW)) + list(zip(model.b, model.gb))
        shapes = list(model.W) + list(model.b)
    m_s = [np.zeros_like(p) for p in shapes]
    v_s = [np.zeros_like(p) for p in shapes]
    for t in range(1, steps + 1):
        idx = r.choice(len(X), batch, replace=False)
        z = model.forward(X[idx])
        z = z - z.max(1, keepdims=True)
        p = np.exp(z) / np.exp(z).sum(1, keepdims=True)
        g = p.copy()
        g[np.arange(batch), yi[idx]] -= 1
        g /= batch
        if isinstance(model, KAN):
            for l in reversed(model.layers):
                g = l.backward(g)
        else:
            model.backward(g)
        for i, (par, gr) in enumerate(params()):
            m_s[i] *= 0.9; m_s[i] += 0.1 * gr
            v_s[i] *= 0.999; v_s[i] += 0.001 * gr ** 2
            par -= lr * (m_s[i] / (1 - 0.9 ** t)) / (np.sqrt(v_s[i] / (1 - 0.999 ** t)) + 1e-8)
    return model


d = load_digits()
Xd = d.data / 16.0 * 2 - 1
Xtr, Xte, ytr_d, yte_d = train_test_split(Xd, d.target, test_size=0.25,
                                          random_state=0, stratify=d.target)

for name, model in [("KAN [64,10], G=8", KAN([64, 10], grid_points=8, span=(-1.1, 1.1), seed=0)),
                    ("MLP [64,64,10]   ", MLP([64, 64, 10], seed=0))]:
    t0 = time.time()
    fit_classifier(model, Xtr, ytr_d)
    acc = (model.forward(Xte).argmax(1) == yte_d).mean()
    print(f"{name}  params {model.n_params:5d}   accuracy {acc:.4f}   {time.time() - t0:6.2f}s")

print()
print("No formula underneath, no advantage. Pixels are not sin(pi x).")

## Exercises

1. **Break the grid.** Set `span=(-0.2, 0.2)` in the `KAN` constructor and remove the re-centring block from `fit`. Watch the model train to a flat line, and explain why in terms of B-spline support.
2. **Overfit with knots.** Sweep `grid_points` from 3 to 64 on 600 training points and plot train and test error. Find where the extra resolution starts to hurt.
3. **Swap the basis.** Replace `BSpline` with sine and cosine harmonics (the SineKAN idea) keeping the same interface. On a target with a high-frequency component, find the harmonic count below which the model cannot represent it at all, and contrast that cliff with the spline basis's smooth degradation.
4. **A three-term formula.** Try `f(x, y, z) = exp(sin(pi x) + y^2) + 0.5 z^3` with a `[3, 1, 1]` KAN. Does it still separate the terms? What happens with `[3, 2, 1]`?
5. **Symbolic snap.** Write a function that takes a learned edge, scores it against a small library of candidate forms (sin, cos, x^2, x^3, exp, log, abs) by affine R-squared, and returns the best match. That is the core of the KAN paper's symbolic regression pipeline.


## Further reading

- Liu, Wang, Vaidya, Ruehle, Halverson, Soljacic, Hou & Tegmark (2024), [KAN: Kolmogorov-Arnold Networks](https://arxiv.org/abs/2404.19756)
- Reinhardt, Ramakrishnan & Gleyzer (2024), [SineKAN: Kolmogorov-Arnold Networks Using Sinusoidal Activation Functions](https://arxiv.org/abs/2407.04149)
- Kolmogorov (1957) and Arnold (1957), the representation theorem the architecture takes its name from
- [The Universal Approximation Theorem in NumPy](https://sesen.ai/blog/universal-approximation-theorem-from-scratch), the sibling theorem that licenses MLPs
